# MV-CSN v2: Multi-View Context-Gated Character Similarity Network (Demo & Results)

Self-contained interactive walk-through of **MV-CSN v2** for Chinese adversarial variant spam detection:
1. **Google Drive Persistence**: Automatic bootstrap from `/content/drive/MyDrive/SYSU_DSE_MVCSN`.
2. **Live Variant Detection & Gate X-Ray**: Per-character obfuscation scores and `[ctx, phon, glyph]` gate weights.
3. **Dynamic OOV Graph Bridging**: Unseen homoglyph / radical-split characters resolved via 32x32 Rendered-Glyph CAE + `pypinyin` priors.
4. **Full Multi-Seed Quantitative Tables**: In-Distribution, Zero-Shot `Clean -> RealVariant`, Per-Attack Breakdown, 8-Row Ablations, and Qwen3 LLM Comparison.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import os
os.chdir(ROOT)
from scripts.gdrive_sync import mount_colab_drive, bootstrap_from_gdrive
mount_colab_drive()
bootstrap_from_gdrive()

from demo.app import build_demo_engine, predict_single_text, PRESET_EXAMPLES
engine = build_demo_engine()
print('Loaded MV-CSN v2 engine | vocab size:', len(engine['vocab']))

[gdrive_sync] Colab drive.mount skipped ('TBE_EPHEM_CREDS_ADDR')


Loaded MV-CSN v2 engine | vocab size: 4557


In [2]:
import numpy as np
for name, txt in PRESET_EXAMPLES.items():
    res = predict_single_text(txt, engine)
    mean_alpha = [round(float(np.mean(res[k])), 3) for k in ('alpha_ctx', 'alpha_phon', 'alpha_glyph')]
    top_obf = sorted([(c, round(p, 3)) for c, p in zip(res['chars'], res['obf_probs']) if p > 0.35], key=lambda x: -x[1])[:5]
    print(f'[{name}]')
    print(f'  pred={res["pred_label"]} (P={res["spam_prob"]:.4f}) | oov={res["oov_resolved"]} | mean_alpha={mean_alpha}')
    print(f'  proc={res["proc_text"]}')
    print(f'  flagged_chars={top_obf}')

[GLYPH (Cross-Script Lookalike Spam)]
  pred=SPAM (P=1.0000) | oov=[] | mean_alpha=[0.912, 0.054, 0.033]
  proc=特惠充100送200え,髙额返利稳赚不赔加微加Q领福利
  flagged_chars=[('え', 0.998), ('髙', 0.996)]
[PHON (Homophone Evasion Spam)]
  pred=SPAM (P=1.0000) | oov=['糸'] | mean_alpha=[0.887, 0.076, 0.037]
  proc=恭喜中大奖,专员联糸您领取现斤红包,请回复确认
  flagged_chars=[('糸', 1.0), ('斤', 0.725)]
[SPLIT (Radical Split & Symbol Bypass)]
  pred=SPAM (P=1.0000) | oov=['讠'] | mean_alpha=[0.811, 0.089, 0.1]
  proc=清联v系客服讠主册即送888元彩金,详询薇信
  flagged_chars=[('讠', 1.0), ('薇', 0.99), ('清', 0.665), ('主', 0.455)]
[MIXED (Multi-Attack Gambling/Loan)]
  pred=SPAM (P=0.9986) | oov=['讠'] | mean_alpha=[0.751, 0.142, 0.107]
  proc=免抵押急用钱?Low利息秒到账,手几号讠主册立抢8折
  flagged_chars=[('讠', 1.0), ('册', 0.829), ('抵', 0.744), ('几', 0.645), ('立', 0.546)]
[CLEAN Ham (Verified Test SMS)]
  pred=HAM (P=0.0000) | oov=[] | mean_alpha=[0.977, 0.019, 0.004]
  proc=我和飞机上一百五十多名旅客眼巴巴期待飞机划破长空腾空而起的时刻
  flagged_chars=[('十', 0.539), ('刻', 0.472), ('巴', 0.384), ('待', 0.3

In [3]:
# Dynamic SpellGCN-style OOV Graph Bridge inspection on unseen lookalike / split chars
vocab = engine['vocab']
bridge = engine['bridge']
oov_map, ext = bridge.build_extended_bundle(['Ꝗえ髙讠'])
for ch in ['え', '髙', 'Ꝗ', '讠']:
    idx = vocab.stoi[ch] if ch in vocab.stoi else oov_map[ch]
    src = 'In-Vocab' if ch in vocab.stoi else f'OOV->{idx}'
    p_nbrs = [(vocab.itos[i], round(float(s), 3)) for i, s in zip(ext['phon_nbr_idx'][idx].tolist(), ext['phon_nbr_sim'][idx].tolist()) if i > 1 and s > 0][:4]
    g_nbrs = [(vocab.itos[i], round(float(s), 3)) for i, s in zip(ext['glyph_nbr_idx'][idx].tolist(), ext['glyph_nbr_sim'][idx].tolist()) if i > 1 and s > 0][:4]
    print(f'{ch} ({src}) | Phonetic Top4: {p_nbrs} | Glyph Top4: {g_nbrs}')

え (In-Vocab) | Phonetic Top4: [] | Glyph Top4: [('ぇ', 0.99), ('无', 0.85), ('ㄤ', 0.821), ('允', 0.819)]
髙 (In-Vocab) | Phonetic Top4: [('皋', 1.0), ('高', 1.0), ('羔', 1.0), ('膏', 1.0)] | Glyph Top4: [('高', 0.944), ('离', 0.896), ('槀', 0.861), ('窝', 0.853)]
Ꝗ (OOV->4557) | Phonetic Top4: [] | Glyph Top4: [('Q', 0.857), ('q', 0.809), ('ɡ', 0.796), ('0', 0.782)]
讠 (OOV->4558) | Phonetic Top4: [('言', 1.0), ('鹽', 1.0), ('顏', 1.0), ('炎', 1.0)] | Glyph Top4: [('ì', 0.939), ('}', 0.926), ('丨', 0.925), ('i', 0.924)]


In [4]:
import json
print('=== Zero-Shot Clean -> RealVariant (Benchmark B) ===')
zs = json.loads(Path('results/zeroshot_real_variants.json').read_text())
for m, s in zs['summary'].items():
    print(f'  {m:18s} | F1={s["f1"]["mean"]:.4f}+-{s["f1"]["std"]:.4f} | AUC={s["auc"]["mean"]:.4f} | HamAcc={s["clean_ham_acc"]["mean"]:.4f}')

print('\n=== 8-Row Ablation Study ===')
abl = json.loads(Path('results/ablations.json').read_text())
for name, s in abl['summary'].items():
    print(f'  {name:28s} | F1={s["f1"]["mean"]:.4f}+-{s["f1"]["std"]:.4f} | dF1={s["delta_f1"]:+.4f}')

print('\n=== Qwen3 Open-Weight LLM vs MV-CSN v2 & Cascade ===')
llm = json.loads(Path('results/llm_comparison.json').read_text())
for m, s in llm['summary'].items():
    ms = s.get('infer_ms_per_sample', {}).get('mean', 0.0)
    p_cnt = llm['params'].get(m, 0)
    print(f'  {m:30s} | F1={s["f1"]["mean"]:.4f} | AUC={s["auc"]["mean"]:.4f} | HamAcc={s["clean_ham_acc"]["mean"]:.4f} | {ms:.3f} ms | params={p_cnt:,}')

=== Zero-Shot Clean -> RealVariant (Benchmark B) ===
  TFIDF-SVM          | F1=0.9801+-0.0000 | AUC=0.9979 | HamAcc=0.9955
  LZASD              | F1=0.9542+-0.0007 | AUC=0.9864 | HamAcc=0.9426
  RBT3               | F1=0.9568+-0.0020 | AUC=0.9987 | HamAcc=0.9970
  MV-CSN v2          | F1=0.9951+-0.0011 | AUC=0.9996 | HamAcc=0.9935
  RBT3 + MV-CSN v2   | F1=0.9964+-0.0011 | AUC=0.9998 | HamAcc=0.9961

=== 8-Row Ablation Study ===
  Full MV-CSN v2               | F1=0.9958+-0.0005 | dF1=+0.0000
  w/o Phonetic View            | F1=0.9962+-0.0006 | dF1=+0.0004
  w/o Glyph Visual View        | F1=0.9961+-0.0009 | dF1=+0.0003
  w/o Context Gate (Uniform)   | F1=0.9950+-0.0003 | dF1=-0.0008
  w/o RoCBert Consistency      | F1=0.9932+-0.0010 | dF1=-0.0026
  w/o Multimodal Anchor        | F1=0.9961+-0.0006 | dF1=+0.0003
  w/o Dynamic OOV Bridge       | F1=0.9952+-0.0006 | dF1=-0.0007
  w/o Split/Noise Preprocessor | F1=0.9706+-0.0063 | dF1=-0.0253

=== Qwen3 Open-Weight LLM vs MV-CSN v2 & Casca